In [1]:
%load_ext autoreload
%autoreload 2

# `Logit` on Orders - Logistic Regression (~1h)

## Select features

🎯 Haydi `wait_time` ve `delay_vs_expected` değişkenlerinin çok `iyi/kötü review`lar üzerindeki etkisini inceleyelim.

👉 `orders` training_set’imizi kullanarak iki adet `multivariate logistic regression` çalıştıracağız:
- `logit_one` → `dim_is_one_star` tahmini için  
- `logit_five` → `dim_is_five_star` tahmini için.

 

In [2]:
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

👉 Dataset’inizi import edin:

In [3]:
from olist.order import Order
orders = Order().get_training_data(with_distance_seller_customer=True)

👉 Kullanmak istediğiniz feature’ları bir listede seçin:

⚠️ Data leakage yaratmadığınızdan emin olun (yani target’tan türetilmiş feature’ları seçmeyin)

💡 `wait_time` ve `delay_vs_expected` değişkenlerinin etkisini anlayabilmek için diğer feature’ların etkisini kontrol etmemiz gerekir, bu yüzden listenize ilgili olabilecek tüm feature’ları dahil edin.

In [4]:
print(orders.columns.tolist())

['order_id', 'wait_time', 'expected_wait_time', 'delay_vs_expected', 'order_status', 'dim_is_five_star', 'dim_is_one_star', 'review_score', 'number_of_items', 'number_of_sellers', 'price', 'freight_value', 'distance_seller_customer']


In [5]:
features = [
    'wait_time',
    'expected_wait_time',
    'delay_vs_expected',
    'number_of_items',
    'number_of_sellers',
    'price',
    'freight_value',
    'distance_seller_customer'
]

orders[features].head()

,wait_time,expected_wait_time,delay_vs_expected,number_of_items,number_of_sellers,price,freight_value,distance_seller_customer
0,8.436574,15.544063,0.0,1,1,29.99,8.72,18.576110
1,13.782037,19.137766,0.0,1,1,118.70,22.76,851.495069
2,9.394213,26.639711,0.0,1,1,159.90,19.22,514.410666
3,13.208750,26.188819,0.0,1,1,45.00,27.20,1822.226336
4,2.873877,12.112049,0.0,1,1,19.90,8.72,29.676625


🕵🏻 Feature’larınızın `multicollinearity` durumunu `VIF index` kullanarak kontrol edin.

* Çok yüksek olmamalıdır (tercihen < 10), böylece partial regression coefficient’larına ve ilgili `p-values` değerlerine güvenebiliriz.
* Verinizi standardize etmeyi unutmayın!
    * Bir `VIF Analysis`, bir feature’ın diğer feature’lara karşı regresyonunu yaparak hesaplanır...
    * Bu yüzden herhangi bir linear regression çalıştırmadan önce feature’ların `scale etkisini kaldırmak` ve eşit öneme sahip olmalarını sağlamak istersiniz!
    
    
📚 <a href="https://www.statisticshowto.com/variance-inflation-factor/">Statistics How To - Variance Inflation Factor</a>

📚  <a href="https://online.stat.psu.edu/stat462/node/180/">PennState - Detecting Multicollinearity Using Variance Inflation Factors</a>

⚖️ Standardize etme:

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X = orders[features].dropna()

X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=features,
    index=X.index
)

X_scaled.head()

,wait_time,expected_wait_time,delay_vs_expected,number_of_items,number_of_sellers,price,freight_value,distance_seller_customer
0,-0.431195,-0.934811,-0.161782,-0.264596,-0.112545,-0.513805,-0.652042,-0.981493
1,0.134174,-0.524874,-0.161782,-0.264596,-0.112545,-0.086641,0.000467,0.423670
2,-0.329909,0.330880,-0.161782,-0.264596,-0.112545,0.111749,-0.164054,-0.145003
3,0.073540,0.279447,-0.161782,-0.264596,-0.112545,-0.441528,0.206816,2.061328
4,-1.019540,-1.326304,-0.161782,-0.264596,-0.112545,-0.562391,-0.652042,-0.962766


👉 Olası multicollinearity durumlarını analiz etmek için VIF Analysis’inizi çalıştırın:

In [8]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()

vif_data["Feature"] = X_scaled.columns

vif_data["VIF"] = [
    variance_inflation_factor(X_scaled.values, i)
    for i in range(X_scaled.shape[1])
]

vif_data


,Feature,VIF
0,wait_time,3.062342
1,expected_wait_time,1.602931
2,delay_vs_expected,2.448919
3,number_of_items,1.371655
4,number_of_sellers,1.095545
5,price,1.208630
6,freight_value,1.680430
7,distance_seller_customer,1.604526


In [9]:
vif_data

,Feature,VIF
0,wait_time,3.062342
1,expected_wait_time,1.602931
2,delay_vs_expected,2.448919
3,number_of_items,1.371655
4,number_of_sellers,1.095545
5,price,1.208630
6,freight_value,1.680430
7,distance_seller_customer,1.604526


## Logistic Regressions

👉 İki adet `Logistic Regression` modeli fit edin:
- `logit_one` → `dim_is_one_star` tahmini için
- `logit_five` → `dim_is_five_star` tahmini için.

`Logit 1️⃣`

In [10]:
# 1 yıldızlı değerlendirmeleri tahmin et
model_data = orders[features + ['dim_is_one_star']].dropna()

X_one = sm.add_constant(model_data[features])
y_one = model_data['dim_is_one_star']

logit_one = sm.Logit(y_one, X_one).fit()

print(logit_one.summary())

Optimization terminated successfully.
         Current function value: 0.272744
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:        dim_is_one_star   No. Observations:                95872
Model:                          Logit   Df Residuals:                    95863
Method:                           MLE   Df Model:                            8
Date:                Thu, 29 Oct 2026   Pseudo R-squ.:                  0.1474
Time:                        12:53:04   Log-Likelihood:                -26149.
converged:                       True   LL-Null:                       -30669.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                       -5.0069      0.073    -68.269      0.000      -5.151

`Logit 5️⃣`

In [11]:
model_data_five = orders[features + ['dim_is_five_star']].dropna()

X_five = sm.add_constant(model_data_five[features])
y_five = model_data_five['dim_is_five_star']

logit_five = sm.Logit(y_five, X_five).fit()

print(logit_five.summary())


Optimization terminated successfully.
         Current function value: 0.636466
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:       dim_is_five_star   No. Observations:                95872
Model:                          Logit   Df Residuals:                    95863
Method:                           MLE   Df Model:                            8
Date:                Thu, 29 Oct 2026   Pseudo R-squ.:                 0.05859
Time:                        12:54:58   Log-Likelihood:                -61019.
converged:                       True   LL-Null:                       -64817.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                        2.3527      0.065     35.941      0.000       2.224

💡 Şimdi bu iki logistic regression’ın sonuçlarını analiz etme zamanı:

- Partial coefficient’ları kendi kelimelerinizle yorumlayın.
- `p-values` kullanarak istatistiksel anlamlılıklarını kontrol edin.
- Coefficient önemleri açısından `logit_one` ve `logit_five` arasında herhangi bir fark görüyor musunuz?

In [14]:
# Aşağıdaki cümlelerden doğru olanları aşağıdaki listeye kaydedin.

a = "delay_vs_expected influences five_star ratings even more than one_star ratings"
b = "wait_time influences five_star ratings even more than one_star"

your_answer = [a]

🧪 __Kodunu Test Et__

In [15]:
from nbresult import ChallengeResult

result = ChallengeResult('logit',
    answers = your_answer
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/turke/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/turke/code/data-logit/data-logit/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_logit.py::TestLogit::test_question PASSED                           [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/logit.pickle

git commit -m 'Completed logit step'

git push origin master



<details>
    <summary>- <i>Açıklamalar ve ileri seviye kavramlar</i> -</summary>


> _Diğer tüm şeyler sabitken, `delay factor`, 1-yıldız review alma ihtimalini etkilemesinden bile daha fazla, 5-yıldızdan mahrum kalma ihtimalini artırma eğilimindedir. Muhtemelen bunun sebebi, 1-yıldız review’ların bizzat çok kötü ürünleri hedeflemesi, kötü teslimatları değil._

❗️ Ancak tamamen titiz olmak için, **iki farklı modelin coefficient’larını karşılaştırırken daha dikkatli olmamız gerekir**, çünkü **benzer popülasyonlara dayanmayabilirler**!
    Burada 2 alt popülasyonumuz var: (1-yıldız verenler ve 5-yıldız verenler) ve bunlar doğaları gereği farklı davranış kalıpları sergileyebilirler. 5-yıldız vermeye daha meyilli “mutlu insanlar”ın, “gecikme” veya “fiyat” söz konusu olduğunda, 1-yıldızı “Lucky-Luke gibi ateşleyen” “huysuz insanlara” göre daha az hassas olmaları gayet mümkün...

</details>



🏁 Tebrikler!

💾 `logit.ipynb` notebook’unuzu commit ve push etmeyi unutmayın!